In [1]:
# data science stack incl gpu tools
from pathlib import Path
import time
import gc

import pandas as pd
import cudf
import cupy as cp

DATA_DIR = Path.home() / "datasets" / "nyc-taxi" / "2025" / "cleaned"

REPORT_DIR = (
    Path.home()
    / "projects"
    / "portfolio"
    / "project-00-spark-validation"
    / "reports"
)

REPORT_DIR.mkdir(exist_ok=True)

files = sorted(
    DATA_DIR.glob("yellow_tripdata_2025-??_clean.parquet")
)

tiers = {
    "1_month": files[:1],
    "3_months": files[:3],
    "6_months": files[:6],
    "12_months": files[:12],
}

timing_cols = [
    "load",
    "filter",
    "null_count",
    "groupby",
    "feature_engineering",
    "sort",
    "total",
]

print("Files found:", len(files))

Files found: 12


In [2]:
# gpu timing
def timed_gpu(func):
    cp.cuda.Stream.null.synchronize()

    start = time.perf_counter()

    result = func()

    cp.cuda.Stream.null.synchronize()

    elapsed = time.perf_counter() - start

    return result, elapsed

In [3]:
# benchmark function with gpu tools
def benchmark_cudf(file_list):
    results = {}

    # 1. Load
    df, results["load"] = timed_gpu(
        lambda: cudf.concat(
            [cudf.read_parquet(str(f)) for f in file_list],
            ignore_index=True
        )
    )

    # 2. Filter
    filtered, results["filter"] = timed_gpu(
        lambda: df[
            (df["trip_distance"] > 0)
            & (df["fare_amount"] > 0)
            & (df["total_amount"] > 0)
        ]
    )

    # 3. Null count
    null_counts, results["null_count"] = timed_gpu(
        lambda: df.isna().sum()
    )

    # 4. Groupby
    grouped, results["groupby"] = timed_gpu(
        lambda: df.groupby("payment_type").agg({
            "VendorID": "count",
            "trip_distance": "mean",
            "fare_amount": "mean",
            "tip_amount": "mean",
            "total_amount": "mean",
        })
    )

    # 5. Feature engineering
    def make_features():
        temp = df.copy()

        temp["trip_duration_min"] = (
            temp["tpep_dropoff_datetime"]
            - temp["tpep_pickup_datetime"]
        ).dt.total_seconds() / 60

        temp["pickup_hour"] = (
            temp["tpep_pickup_datetime"].dt.hour
        )

        tip_pct = (
            temp["tip_amount"]
            / temp["fare_amount"]
            * 100
        )

        temp["tip_pct"] = tip_pct.where(
            temp["fare_amount"] > 0
        )

        return temp

    engineered, results["feature_engineering"] = timed_gpu(
        make_features
    )

    # 6. Sort
    sorted_df, results["sort"] = timed_gpu(
        lambda: df.sort_values(
            "total_amount",
            ascending=False
        )
    )

    results["rows"] = len(df)

    results["total"] = sum(
        results[key]
        for key in timing_cols[:-1]
    )

    return results

In [4]:
# benchmark per teir
def run_repeated_gpu_benchmark(
    benchmark_func,
    tiers,
    backend="cuDF",
    runs=5
):
    records = []

    for tier_name, tier_files in tiers.items():
        print(f"\n{backend} — {tier_name}")

        for run_num in range(1, runs + 1):
            gc.collect()
            cp.get_default_memory_pool().free_all_blocks()

            result = benchmark_func(tier_files)

            result["tier"] = tier_name
            result["backend"] = backend
            result["run"] = run_num

            records.append(result)

            print(
                f"Run {run_num}: "
                f"{result['total']:.4f} s"
            )

    return pd.DataFrame(records)

In [5]:
# january single month test
test_gpu = benchmark_cudf(tiers["1_month"])
test_gpu

{'load': 0.6080352340359241,
 'filter': 0.047551235067658126,
 'null_count': 0.043992080027237535,
 'groupby': 0.025630700052715838,
 'feature_engineering': 0.20637530996464193,
 'sort': 0.0733597599901259,
 'rows': 3475082,
 'total': 1.0049443191383034}

In [6]:
# row confirm
test_gpu["rows"]

3475082

In [7]:
# benchmark 5 rep
cudf_runs = run_repeated_gpu_benchmark(
    benchmark_cudf,
    tiers,
    backend="cuDF",
    runs=5
)

cudf_runs


cuDF — 1_month
Run 1: 0.3016 s
Run 2: 0.2734 s
Run 3: 0.2725 s
Run 4: 0.2734 s
Run 5: 0.2697 s

cuDF — 3_months
Run 1: 0.7917 s
Run 2: 0.7897 s
Run 3: 0.7967 s
Run 4: 0.7910 s
Run 5: 0.7923 s

cuDF — 6_months
Run 1: 1.6513 s
Run 2: 1.6473 s
Run 3: 1.6409 s
Run 4: 1.6515 s
Run 5: 1.6553 s

cuDF — 12_months
Run 1: 3.2715 s
Run 2: 3.2668 s
Run 3: 3.2972 s
Run 4: 3.3058 s
Run 5: 3.3087 s


,load,filter,null_count,groupby,feature_engineering,sort,rows,total,tier,backend,run
0,0.116276,0.035162,0.021577,0.006363,0.059172,0.063094,3475082,0.301643,1_month,cuDF,1
1,0.086407,0.034995,0.021629,0.008221,0.058970,0.063141,3475082,0.273364,1_month,cuDF,2
2,0.085652,0.035294,0.021501,0.008125,0.058948,0.062969,3475082,0.272489,1_month,cuDF,3
3,0.086279,0.035112,0.021828,0.008598,0.058660,0.062965,3475082,0.273441,1_month,cuDF,4
4,0.085409,0.035019,0.021651,0.006184,0.058653,0.062831,3475082,0.269747,1_month,cuDF,5
5,0.263247,0.098806,0.045868,0.015765,0.158431,0.209572,11197681,0.791690,3_months,cuDF,1
6,0.260776,0.098502,0.046056,0.015782,0.157799,0.210833,11197681,0.789748,3_months,cuDF,2
7,0.262397,0.098668,0.046024,0.021653,0.157447,0.210549,11197681,0.796737,3_months,cuDF,3
8,0.261526,0.098507,0.046010,0.015813,0.158051,0.211103,11197681,0.791010,3_months,cuDF,4
9,0.261508,0.099571,0.046330,0.015936,0.158457,0.210495,11197681,0.792297,3_months,cuDF,5


In [8]:
#confirm dataframe
cudf_runs.shape

(20, 11)

In [9]:
# export
cudf_runs.to_csv(
    REPORT_DIR / "cudf_benchmark_runs.csv",
    index=False
)

In [10]:
cudf_summary = (
    cudf_runs
    .groupby("tier")[timing_cols]
    .agg(["median", "mean", "std"])
)

cudf_summary

load                        filter                      \
             median      mean       std    median      mean       std   
tier                                                                    
12_months  1.091773  1.091462  0.001884  0.393350  0.393432  0.000681   
1_month    0.085836  0.092624  0.015284  0.034905  0.034982  0.000192   
3_months   0.261370  0.261525  0.000927  0.098757  0.098931  0.000477   
6_months   0.550388  0.551151  0.001645  0.200331  0.200065  0.000721   

          null_count                       groupby  ...            \
              median      mean       std    median  ...       std   
tier                                                ...             
12_months   0.137593  0.137584  0.000533  0.062991  ...  0.013935   
1_month     0.021809  0.021862  0.000188  0.008187  ...  0.001212   
3_months    0.046204  0.046221  0.000074  0.015917  ...  0.003746   
6_months    0.077854  0.077919  0.000350  0.031758  ...  0.010923   

          feature_engineering                          sort            \
                       median      mean       std    median      mean   
tier                                                                    
12_months            0.633408  0.632976  0.003410  0.939807  0.939937   
1_month              0.058900  0.059038  0.000262  0.062990  0.063086   
3_months             0.158173  0.157933  0.000440  0.210733  0.211012   
6_months             0.317399  0.317900  0.001348  0.457390  0.457720   

                        total                      
                std    median      mean       std  
tier                                               
12_months  0.001313  3.259784  3.267826  0.015166  
1_month    0.000186  0.272826  0.279362  0.016777  
3_months   0.000901  0.792562  0.793205  0.004726  
6_months   0.000842  1.638407  1.641324  0.008736  

[4 rows x 21 columns]

In [11]:
cudf_summary.to_csv(
    REPORT_DIR / "cudf_benchmark_summary.csv"
)